# LeetCode 1502 — Can Make Arithmetic Progression From Sequence bằng Python

**Link đề:** https://leetcode.com/problems/can-make-arithmetic-progression-from-sequence/
**Độ khó:** Easy · **Chủ đề:** Array, Sorting, Hash Table (cho lời giải `O(n)`)

```python
class Solution:
    def canMakeArithmeticProgression(self, arr: list[int]) -> bool:
        ...
```

### Mục tiêu học
1. Hiểu **cấp số cộng** (arithmetic progression — AP): dãy có hiệu hai số liên tiếp không đổi.
2. Thấy vì sao thử mọi hoán vị là bất khả thi, và **insight**: một AP sắp xếp lại luôn có thứ tự tăng (hoặc giảm).
3. Viết lời giải chính: sort rồi kiểm tra các hiệu liên tiếp, dùng `zip`, `all`, `range`.
4. Nâng cao: lời giải `O(n)` không sort bằng `min`, `max`, công sai `d` và `set`.
5. Trace, chứng minh, phân tích, test đối chiếu và tự giải biến thể.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Một dãy số gọi là **cấp số cộng** nếu hiệu giữa **mọi cặp phần tử liên tiếp** đều bằng nhau.

Cho mảng `arr`, trả về `True` nếu có thể **sắp xếp lại** (đổi thứ tự tùy ý) mảng để tạo thành cấp số cộng; ngược lại `False`.

| `arr` | Sắp xếp lại | Hiệu liên tiếp | Output |
|---|---|---|---|
| `[3,5,1]` | `[1,3,5]` (hoặc `[5,3,1]`) | `2, 2` (hoặc `-2, -2`) | `True` |
| `[1,2,4]` | không cách nào | `1, 2` | `False` |

**Constraints:** `2 <= arr.length <= 1000`, `-10^6 <= arr[i] <= 10^6`.

Lưu ý:
- Mảng có thể có **số trùng**: `[7,7,7]` là AP với công sai `0` → `True`. Nhưng `[1,1,2]` → `False`.
- Mảng có **2 phần tử** luôn là AP (chỉ có một hiệu).

## 2. Nền tảng: cấp số cộng và công sai

Dãy `a0, a1, ..., a(n-1)` là AP với **công sai** (common difference) `d` nếu

```
a1 - a0 = a2 - a1 = ... = a(n-1) - a(n-2) = d
```

Tương đương: `a_k = a0 + k * d`.

Ví dụ: `1, 3, 5, 7` (d = 2); `10, 7, 4` (d = -3); `5, 5, 5` (d = 0).

In [1]:
def diffs(seq):
    return [seq[i + 1] - seq[i] for i in range(len(seq) - 1)]

for seq in [[1, 3, 5, 7], [10, 7, 4], [5, 5, 5], [1, 2, 4]]:
    print(f"{str(seq):<14} hiệu liên tiếp = {diffs(seq)}")

[1, 3, 5, 7]   hiệu liên tiếp = [2, 2, 2]
[10, 7, 4]     hiệu liên tiếp = [-3, -3]
[5, 5, 5]      hiệu liên tiếp = [0, 0]
[1, 2, 4]      hiệu liên tiếp = [1, 2]


`range(len(seq) - 1)` dừng ở chỉ số áp chót vì ta đọc `seq[i + 1]` — nếu `i` chạy tới phần tử cuối thì `seq[i+1]` vượt biên.

### 2.1. `zip(a, a[1:])` — duyệt các cặp kề nhau

`zip` ghép hai dãy thành từng cặp; `a[1:]` là `a` bỏ phần tử đầu. Ghép lại được các cặp `(a[0], a[1]), (a[1], a[2]), ...` — gọn và không lo vượt biên (zip dừng ở dãy ngắn hơn).

In [2]:
a = [1, 3, 5, 7]
print(list(zip(a, a[1:])))
print([y - x for x, y in zip(a, a[1:])])

[(1, 3), (3, 5), (5, 7)]
[2, 2, 2]


### 2.2. `all(...)` — "tất cả đều đúng?"

`all(dãy)` trả `True` nếu **mọi** phần tử đúng (và `True` với dãy rỗng). Nó **dừng sớm** ngay khi gặp phần tử sai.

In [3]:
print(all([True, True, True]))
print(all([True, False, True]))
print(all([]))                      # dãy rỗng -> True
a = [1, 3, 5, 8]
d = a[1] - a[0]
print("mọi hiệu bằng d?", all(y - x == d for x, y in zip(a, a[1:])))

True
False
True
mọi hiệu bằng d? False


### 2.3. `sorted` vs `.sort()`

`sorted(arr)` tạo list mới đã sắp xếp; `arr.sort()` sắp xếp tại chỗ và trả `None`. Bài này dùng `sorted` để không sửa input.

## 3. Cách 0 — Ngây thơ: thử mọi hoán vị (chỉ để hiểu, KHÔNG nộp)

"Có thể sắp xếp lại" → thử tất cả thứ tự. `itertools.permutations(arr)` sinh mọi hoán vị. Có `n!` hoán vị — với `n = 10` đã là 3,6 triệu, `n = 1000` thì không tưởng.

In [4]:
from itertools import permutations
import math

def can_make_ap_permutations(arr):
    for p in permutations(arr):
        if all(y - x == p[1] - p[0] for x, y in zip(p, p[1:])):
            return True
    return False

print(can_make_ap_permutations([3, 5, 1]))   # True
print(can_make_ap_permutations([1, 2, 4]))   # False
for n in [5, 10, 20]:
    print(f"n={n}: {math.factorial(n):,} hoán vị")

True
False
n=5: 120 hoán vị
n=10: 3,628,800 hoán vị
n=20: 2,432,902,008,176,640,000 hoán vị


## 4. Nút thắt & Insight — AP luôn là dãy đã sắp xếp

Nếu `d > 0` thì `a0 < a1 < a2 < ...` (tăng); `d < 0` thì giảm; `d = 0` thì hằng. Tức là **một AP luôn đơn điệu**. Đảo ngược một AP giảm cũng là một AP (công sai `-d`).

Vậy: *nếu `arr` sắp xếp lại được thành AP, thì chính dãy `sorted(arr)` phải là AP.* Ta không cần thử `n!` thứ tự — chỉ cần thử **đúng một** thứ tự: thứ tự tăng dần.

Đó là gợi ý của đề: *"mọi AP hợp lệ đều ở dạng đã sắp xếp — sort rồi kiểm tra hiệu liên tiếp"*.

In [5]:
arr = [3, 5, 1]
s = sorted(arr)
print("sorted:", s, "| hiệu:", diffs(s))
arr = [1, 2, 4]
s = sorted(arr)
print("sorted:", s, "| hiệu:", diffs(s))

sorted: [1, 3, 5] | hiệu: [2, 2]
sorted: [1, 2, 4] | hiệu: [1, 2]


## 5. Cách 1 — Sort + kiểm tra hiệu (lời giải chính)

Mổ xẻ:
1. `s = sorted(arr)`.
2. `d = s[1] - s[0]` — công sai "ứng viên" (mảng có ít nhất 2 phần tử nên an toàn).
3. Với mọi `i` từ 2 tới `n-1`: nếu `s[i] - s[i-1] != d` → `False`.
4. Hết vòng → `True`.

In [6]:
def can_make_ap_sort(arr: list[int]) -> bool:
    s = sorted(arr)
    d = s[1] - s[0]
    for i in range(2, len(s)):
        if s[i] - s[i - 1] != d:
            return False
    return True

print(can_make_ap_sort([3, 5, 1]))       # True
print(can_make_ap_sort([1, 2, 4]))       # False
print(can_make_ap_sort([7, 7, 7]))       # True (d = 0)
print(can_make_ap_sort([1, 1, 2]))       # False

True
False
True
False


## 6. TRACE từng bước

Ví dụ `arr = [9, 1, 5, 3, 7]` → `s = [1, 3, 5, 7, 9]`, `d = 2`:

| i | s[i-1] | s[i] | hiệu | == d? |
|---:|---:|---:|---:|:-:|
| 2 | 3 | 5 | 2 | ✅ |
| 3 | 5 | 7 | 2 | ✅ |
| 4 | 7 | 9 | 2 | ✅ |

→ `True`.

Ví dụ `arr = [4, 1, 10, 7, 12]` → `s = [1, 4, 7, 10, 12]`, `d = 3`: tại `i = 4`, hiệu `12 - 10 = 2 ≠ 3` → `False`.

In [7]:
def trace_ap_sort(arr):
    s = sorted(arr)
    d = s[1] - s[0]
    print(f"arr = {arr} -> sorted = {s}, d = s[1] - s[0] = {d}")
    for i in range(2, len(s)):
        diff = s[i] - s[i - 1]
        ok = diff == d
        print(f"  i={i}: {s[i]} - {s[i-1]} = {diff} {'==' if ok else '!='} {d}")
        if not ok:
            print("  => False")
            return False
    print("  => True")
    return True

trace_ap_sort([9, 1, 5, 3, 7])
print()
trace_ap_sort([4, 1, 10, 7, 12])

arr = [9, 1, 5, 3, 7] -> sorted = [1, 3, 5, 7, 9], d = s[1] - s[0] = 2
  i=2: 5 - 3 = 2 == 2
  i=3: 7 - 5 = 2 == 2
  i=4: 9 - 7 = 2 == 2
  => True

arr = [4, 1, 10, 7, 12] -> sorted = [1, 4, 7, 10, 12], d = s[1] - s[0] = 3
  i=2: 7 - 4 = 3 == 3
  i=3: 10 - 7 = 3 == 3
  i=4: 12 - 10 = 2 != 3
  => False


False

## 7. Cách 1b — Viết gọn bằng `zip` + `all` hoặc `set` các hiệu

- `all(...)`: như trên.
- Hoặc: tập các hiệu liên tiếp có **đúng 1 phần tử** ⇔ mọi hiệu bằng nhau.

In [8]:
def can_make_ap_all(arr):
    s = sorted(arr)
    d = s[1] - s[0]
    return all(b2 - a2 == d for a2, b2 in zip(s, s[1:]))

def can_make_ap_diffset(arr):
    s = sorted(arr)
    return len({b2 - a2 for a2, b2 in zip(s, s[1:])}) == 1

for arr in [[3, 5, 1], [1, 2, 4], [7, 7, 7]]:
    print(arr, can_make_ap_all(arr), can_make_ap_diffset(arr))

[3, 5, 1] True True
[1, 2, 4] False False
[7, 7, 7] True True


`{biểu_thức for ...}` là **set comprehension** — giống list comprehension nhưng tạo set (tự bỏ trùng).

## 8. Vì sao cách sort đúng?

**Mệnh đề:** `arr` sắp xếp lại được thành AP ⇔ `sorted(arr)` là AP.

- (⇐) Hiển nhiên: `sorted(arr)` chính là một cách sắp xếp lại.
- (⇒) Giả sử có cách sắp xếp `b` là AP với công sai `d`. Nếu `d >= 0` thì `b` không giảm, nên `b` chính là `sorted(arr)` (dãy không giảm của cùng một tập phần tử là duy nhất). Nếu `d < 0` thì `b` giảm nghiêm ngặt; đảo ngược `b` được dãy tăng, là AP với công sai `-d`, và cũng chính là `sorted(arr)`.

Vòng lặp kiểm tra đúng định nghĩa AP trên `sorted(arr)`: trả `False` ngay khi có hiệu khác `d`, trả `True` khi mọi hiệu bằng `d`.

## 9. Code nộp LeetCode — sort

In [9]:
class Solution:
    def canMakeArithmeticProgression(self, arr: list[int]) -> bool:
        s = sorted(arr)
        d = s[1] - s[0]
        for i in range(2, len(s)):
            if s[i] - s[i - 1] != d:
                return False
        return True

sol = Solution()
assert sol.canMakeArithmeticProgression([3, 5, 1]) is True
assert sol.canMakeArithmeticProgression([1, 2, 4]) is False
print("PASS ví dụ đề")

PASS ví dụ đề


## 10. Cách 2 — `O(n)` không cần sort: min, max, công sai và set

Nếu dãy là AP với `n` phần tử thì:
- phần tử nhỏ nhất `lo` và lớn nhất `hi` là hai đầu,
- công sai `d = (hi - lo) / (n - 1)` — **bắt buộc chia hết**,
- các phần tử phải đúng là `lo, lo + d, lo + 2d, ..., hi`, **mỗi số một lần** (khi `d ≠ 0`).

Từng bước:
1. `lo, hi = min(arr), max(arr)`. Nếu `lo == hi` → mọi số bằng nhau → `True`.
2. Nếu `(hi - lo) % (n - 1) != 0` → `False`.
3. `d = (hi - lo) // (n - 1)`; với mỗi `x`: `(x - lo) % d` phải bằng 0 (x nằm "trên lưới").
4. Số phần tử phân biệt phải đúng bằng `n` (không trùng) — dùng `set`.

In [10]:
def can_make_ap_linear(arr: list[int]) -> bool:
    n = len(arr)
    lo, hi = min(arr), max(arr)
    if lo == hi:                       # toàn bộ bằng nhau: d = 0
        return True
    if (hi - lo) % (n - 1) != 0:       # công sai phải là số nguyên
        return False
    d = (hi - lo) // (n - 1)
    seen = set()
    for x in arr:
        if (x - lo) % d != 0:          # x không nằm trên lưới lo + k*d
            return False
        if x in seen:                  # d != 0 nên không được trùng
            return False
        seen.add(x)
    return True                        # n số phân biệt, đều thuộc {lo, lo+d, ..., hi} (đúng n giá trị)

for arr in [[3, 5, 1], [1, 2, 4], [7, 7, 7], [1, 1, 2], [0, 0, 3, 3], [1, 3, 3, 7]]:
    print(f"{str(arr):<14} -> {can_make_ap_linear(arr)} | sort cho: {can_make_ap_sort(arr)}")

[3, 5, 1]      -> True | sort cho: True
[1, 2, 4]      -> False | sort cho: False
[7, 7, 7]      -> True | sort cho: True
[1, 1, 2]      -> False | sort cho: False
[0, 0, 3, 3]   -> False | sort cho: False
[1, 3, 3, 7]   -> False | sort cho: False


### Trace cách 2

`arr = [1, 7, 4, 10]`: `lo = 1`, `hi = 10`, `n - 1 = 3`, `(10 - 1) % 3 = 0` → `d = 3`. Lưới là `{1, 4, 7, 10}`.

| x | (x - lo) % d | trong seen? | seen sau |
|---:|:-:|:-:|---|
| 1 | 0 | không | {1} |
| 7 | 0 | không | {1, 7} |
| 4 | 0 | không | {1, 7, 4} |
| 10 | 0 | không | {1, 7, 4, 10} |

→ `True`.

In [11]:
def trace_ap_linear(arr):
    n = len(arr)
    lo, hi = min(arr), max(arr)
    print(f"arr = {arr}: lo = {lo}, hi = {hi}, n - 1 = {n - 1}")
    if lo == hi:
        print("  lo == hi -> True"); return True
    if (hi - lo) % (n - 1):
        print(f"  ({hi} - {lo}) % {n - 1} = {(hi - lo) % (n - 1)} != 0 -> False"); return False
    d = (hi - lo) // (n - 1)
    print(f"  d = {d}, lưới = {list(range(lo, hi + 1, d))}")
    seen = set()
    for x in arr:
        r = (x - lo) % d
        dup = x in seen
        print(f"  x={x:>3}: (x - lo) % d = {r}, đã có? {dup}")
        if r or dup:
            print("  => False"); return False
        seen.add(x)
    print("  => True"); return True

trace_ap_linear([1, 7, 4, 10])
print()
trace_ap_linear([1, 3, 3, 7])     # chia hết, mọi số đúng lưới, nhưng trùng 3 (thiếu 5)
print()
trace_ap_linear([0, 1, 3])        # (3 - 0) % 2 != 0 -> công sai không nguyên

arr = [1, 7, 4, 10]: lo = 1, hi = 10, n - 1 = 3
  d = 3, lưới = [1, 4, 7, 10]
  x=  1: (x - lo) % d = 0, đã có? False
  x=  7: (x - lo) % d = 0, đã có? False
  x=  4: (x - lo) % d = 0, đã có? False
  x= 10: (x - lo) % d = 0, đã có? False
  => True

arr = [1, 3, 3, 7]: lo = 1, hi = 7, n - 1 = 3
  d = 2, lưới = [1, 3, 5, 7]
  x=  1: (x - lo) % d = 0, đã có? False
  x=  3: (x - lo) % d = 0, đã có? False
  x=  3: (x - lo) % d = 0, đã có? True
  => False

arr = [0, 1, 3]: lo = 0, hi = 3, n - 1 = 2
  (3 - 0) % 2 = 1 != 0 -> False


False

### Vì sao cách 2 đúng?

- Nếu `arr` là AP (sau khi sắp xếp) với `d ≠ 0`, các phần tử là `lo + k*d` với `k = 0..n-1`, phân biệt, và `hi - lo = (n-1)d` → mọi kiểm tra đều qua.
- Ngược lại, nếu mọi kiểm tra qua: có `n` số **phân biệt**, mỗi số thuộc tập `{lo + k*d : k = 0..n-1}` (vì nằm trên lưới và trong đoạn `[lo, hi]`). Tập này có đúng `n` phần tử → `arr` chiếm **toàn bộ** tập đó → sắp xếp lại là AP.

Python: `%` với số âm cho kết quả không âm khi số chia dương (`-3 % 3 == 0`, `-2 % 3 == 1`), và ở đây `x - lo >= 0` nên không lo dấu.

## 11. Code nộp LeetCode — `O(n)`

In [12]:
class SolutionLinear:
    def canMakeArithmeticProgression(self, arr: list[int]) -> bool:
        n = len(arr)
        lo, hi = min(arr), max(arr)
        if lo == hi:
            return True
        if (hi - lo) % (n - 1):
            return False
        d = (hi - lo) // (n - 1)
        seen = set()
        for x in arr:
            if (x - lo) % d or x in seen:
                return False
            seen.add(x)
        return True

assert SolutionLinear().canMakeArithmeticProgression([3, 5, 1]) is True
assert SolutionLinear().canMakeArithmeticProgression([1, 2, 4]) is False
print("SolutionLinear PASS ví dụ đề (đổi tên thành Solution khi nộp)")

SolutionLinear PASS ví dụ đề (đổi tên thành Solution khi nộp)


`if (hi - lo) % (n - 1):` — số khác 0 được coi là **truthy** (đúng), bằng 0 là **falsy**. Viết vậy tương đương `!= 0`.

## 12. Phân tích Time / Space

Gọi `n = len(arr)`.

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 0. Hoán vị | `O(n · n!)` | `O(n)` | chỉ để hiểu |
| 1. Sort + hiệu | `O(n log n)` | `O(n)` (`sorted` tạo list mới) | dễ, chuẩn |
| 2. min/max + set | `O(n)` trung bình | `O(n)` | nhanh hơn về lý thuyết |

Có thể giảm bộ nhớ cách 1 xuống `O(1)` phụ bằng `arr.sort()` tại chỗ (nhưng sửa input). Với `n <= 1000`, cả cách 1 và 2 đều tức thì.

## 13. Kiểm thử: mẫu đề, biên, phản ví dụ, đối chiếu

In [13]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<36} -> {actual}")

tests = [
    ("đề 1",                               [3, 5, 1],            True),
    ("đề 2",                               [1, 2, 4],            False),
    ("2 phần tử",                          [10, -10],            True),
    ("toàn bằng nhau (d = 0)",             [7, 7, 7],            True),
    ("trùng nhưng không hằng",             [1, 1, 2],            False),
    ("công sai âm khi sắp giảm",           [10, 7, 4, 1],        True),
    ("số âm",                              [-5, -1, -3, 1],      True),
    ("không chia hết công sai",            [0, 1, 3],            False),
    ("đúng lưới nhưng trùng (bẫy set)",    [1, 3, 3, 7],         False),
    ("chia hết nhưng lệch lưới",           [0, 2, 3, 6],         False),
    ("biên giá trị",                       [-10**6, 10**6, 0],   True),
]
for name, arr, exp in tests:
    for f in (can_make_ap_sort, can_make_ap_all, can_make_ap_diffset, can_make_ap_linear):
        assert f(arr) == exp, (f.__name__, arr)
    assert SolutionLinear().canMakeArithmeticProgression(arr) == exp
    check(name, Solution().canMakeArithmeticProgression(arr), exp)

PASS  đề 1                                 -> True
PASS  đề 2                                 -> False
PASS  2 phần tử                            -> True
PASS  toàn bằng nhau (d = 0)               -> True
PASS  trùng nhưng không hằng               -> False
PASS  công sai âm khi sắp giảm             -> True
PASS  số âm                                -> True
PASS  không chia hết công sai              -> False
PASS  đúng lưới nhưng trùng (bẫy set)      -> False
PASS  chia hết nhưng lệch lưới             -> False
PASS  biên giá trị                         -> True


In [14]:
# Đối chiếu ngẫu nhiên, kể cả với brute force hoán vị (n nhỏ)
import random
random.seed(2026)
for _ in range(1500):
    n = random.randint(2, 6)
    if random.random() < 0.5:                       # sinh AP thật rồi xáo trộn
        a0, d = random.randint(-20, 20), random.randint(-5, 5)
        arr = [a0 + k * d for k in range(n)]
        if random.random() < 0.3:                   # làm hỏng một phần tử
            arr[random.randrange(n)] += random.choice([-1, 1])
        random.shuffle(arr)
    else:
        arr = [random.randint(-6, 6) for _ in range(n)]
    exp = can_make_ap_permutations(arr)
    assert can_make_ap_sort(arr) == exp, arr
    assert can_make_ap_linear(arr) == exp, arr
print("1500 mảng ngẫu nhiên: sort và O(n) khớp brute force hoán vị. PASS")

1500 mảng ngẫu nhiên: sort và O(n) khớp brute force hoán vị. PASS


`random.shuffle(arr)` xáo trộn list **tại chỗ** — giúp test rằng thuật toán không phụ thuộc thứ tự ban đầu.

## 14. Lỗi người mới hay gặp

### Lỗi 1: Không sắp xếp, kiểm tra trực tiếp trên `arr`
```python
d = arr[1] - arr[0]   # SAI: [3,5,1] có hiệu 2, -4
```

### Lỗi 2: Dùng `/` cho công sai trong cách `O(n)` rồi so float
`(hi - lo) / (n - 1)` cho float; có thể dùng được nhưng dễ dính sai số và quên kiểm tra chia hết. Hãy kiểm `%` trước rồi dùng `//`.

### Lỗi 3: Cách `O(n)` quên kiểm tra **trùng**
Ví dụ `[1, 3, 3, 7]`: `lo = 1`, `hi = 7`, `d = 2`, mọi số đều nằm trên lưới `{1, 3, 5, 7}` — nhưng có hai số `3` và thiếu `5` → phải `False`. Không có `set` sẽ trả `True` sai.

### Lỗi 4: Chia cho `d = 0`
Khi mọi phần tử bằng nhau, `d = 0` → `(x - lo) % 0` báo `ZeroDivisionError`. Phải xử lý `lo == hi` trước.

### Lỗi 5: Bắt đầu vòng kiểm tra sai chỉ số → đọc vượt biên `s[i + 1]` ở phần tử cuối.

In [15]:
def wrong_no_sort(arr):
    d = arr[1] - arr[0]
    return all(arr[i] - arr[i - 1] == d for i in range(2, len(arr)))

def wrong_linear_no_dup(arr):
    n = len(arr); lo, hi = min(arr), max(arr)
    if lo == hi: return True
    if (hi - lo) % (n - 1): return False
    d = (hi - lo) // (n - 1)
    return all((x - lo) % d == 0 for x in arr)     # lỗi 3: không kiểm trùng

def wrong_linear_zero(arr):
    n = len(arr); lo, hi = min(arr), max(arr)
    d = (hi - lo) // (n - 1)
    return all((x - lo) % d == 0 for x in arr)     # lỗi 4

print("lỗi 1 [3,5,1]  :", wrong_no_sort([3, 5, 1]), "| đúng:", can_make_ap_sort([3, 5, 1]))
print("lỗi 3 [1,3,3,7]:", wrong_linear_no_dup([1, 3, 3, 7]), "| đúng:", can_make_ap_sort([1, 3, 3, 7]))
try:
    wrong_linear_zero([7, 7, 7])
except ZeroDivisionError as e:
    print("lỗi 4 [7,7,7]  : ZeroDivisionError:", e)

lỗi 1 [3,5,1]  : False | đúng: True
lỗi 3 [1,3,3,7]: True | đúng: False
lỗi 4 [7,7,7]  : ZeroDivisionError: integer modulo by zero


## 15. Mẹo nhận dạng pattern

- "Có thể **sắp xếp lại** để thỏa tính chất X?" → thường chỉ cần thử **một** thứ tự chuẩn (đã sort). Đừng thử mọi hoán vị.
- "Kiểm tra mọi cặp **kề nhau**" → `zip(a, a[1:])` hoặc `range(1, n)`.
- "Dãy đều / cấp số cộng" → biết `lo`, `hi`, `n` là suy ra được công sai `(hi - lo) / (n - 1)` → có lời giải `O(n)` bằng set.
- Họ bài: 1630 Arithmetic Subarrays (kiểm tra nhiều đoạn con), 1228 Missing Number In Arithmetic Progression, 413 Arithmetic Slices.

### Biến thể

### Biến thể A — Missing Number in AP (LeetCode 1228)
Một AP (đã giữ nguyên thứ tự) bị mất **một** phần tử không ở hai đầu. Tìm nó: công sai thật là `(last - first) / n` (vì đầy đủ có `n + 1` phần tử).

In [16]:
def missing_in_ap(arr):
    n = len(arr)
    d = (arr[-1] - arr[0]) // n
    expected = arr[0]
    for x in arr:
        if x != expected:
            return expected
        expected += d
    return arr[0]          # d = 0: không thiếu gì đặc biệt

print(missing_in_ap([5, 7, 11, 13]))    # 9
print(missing_in_ap([15, 13, 12]))      # 14

9
14


### Biến thể B — Arithmetic Subarrays (LeetCode 1630)
Với nhiều truy vấn `[l, r]`, hỏi đoạn `arr[l..r]` có sắp lại thành AP không → gọi lại hàm của ta trên từng đoạn con.

In [17]:
def check_arithmetic_subarrays(nums, l, r):
    return [can_make_ap_linear(nums[a:b2 + 1]) for a, b2 in zip(l, r)]

print(check_arithmetic_subarrays([4, 6, 5, 9, 3, 7], [0, 0, 2], [2, 3, 5]))   # [True, False, True]

[True, False, True]


## 16. Luyện tập — 6 bài từ dễ đến nâng cao

### Bài 1 (tính tay)
`arr = [20, 5, 15, 10, 0]` có sắp lại thành AP không? Công sai là bao nhiêu? Trả về tuple `(kết_quả, công_sai_khi_sắp_tăng)`.

In [18]:
def exercise_by_hand() -> tuple[bool, int]:
    # TODO
    pass

# assert exercise_by_hand() == (True, 5)

### Bài 2 (điền chỗ trống — cách sort)

In [19]:
def exercise_can_make_ap(arr: list[int]) -> bool:
    s = sorted(arr)
    # TODO 1: d = ?
    # TODO 2: duyệt i từ 2, nếu hiệu khác d -> False
    pass

# assert exercise_can_make_ap([3, 5, 1]) is True
# assert exercise_can_make_ap([1, 2, 4]) is False

### Bài 3 (dãy đã cho có là AP không — KHÔNG được sắp lại)

In [20]:
def exercise_is_ap_as_is(arr: list[int]) -> bool:
    # TODO: dùng zip + all
    pass

# assert exercise_is_ap_as_is([5, 3, 1]) is True
# assert exercise_is_ap_as_is([3, 5, 1]) is False

### Bài 4 (cấp số nhân)
Kiểm tra có sắp lại `arr` (toàn số **dương**) thành **cấp số nhân** (tỉ số hai số liên tiếp không đổi) không. Gợi ý: tránh chia float — dùng phép nhân chéo.

Ví dụ `[4, 1, 16, 2, 8]` → `True` (1,2,4,8,16); `[1, 2, 3]` → `False`.

In [21]:
def exercise_can_make_gp(arr: list[int]) -> bool:
    # TODO
    pass

# assert exercise_can_make_gp([4, 1, 16, 2, 8]) is True
# assert exercise_can_make_gp([1, 2, 3]) is False
# assert exercise_can_make_gp([3, 3, 3]) is True

### Bài 5 (cách `O(n)` tự viết)
Viết lại cách 2 không nhìn mục 10.

In [22]:
def exercise_can_make_ap_linear(arr: list[int]) -> bool:
    # TODO: lo, hi, kiểm tra chia hết, d, lưới, trùng
    pass

# assert exercise_can_make_ap_linear([1, 7, 4, 10]) is True
# assert exercise_can_make_ap_linear([1, 3, 3, 7]) is False
# assert exercise_can_make_ap_linear([2, 2]) is True

### Bài 6 (mini challenge — đoạn AP dài nhất)
Cho `arr` (không sắp lại), trả về độ dài **đoạn con liên tiếp** dài nhất là AP. Mảng 1 hoặc 2 phần tử: chính độ dài đó.

Ví dụ `[1, 3, 5, 7, 2, 4, 6]` → `4` (`1,3,5,7`); `[10, 7, 4, 6, 8, 10, 11]` → `4` (`4,6,8,10`).

In [23]:
def exercise_longest_ap_subarray(arr: list[int]) -> int:
    # TODO: duyệt một lần, theo dõi hiệu hiện tại và độ dài đoạn hiện tại
    pass

# assert exercise_longest_ap_subarray([1, 3, 5, 7, 2, 4, 6]) == 4
# assert exercise_longest_ap_subarray([10, 7, 4, 6, 8, 10, 11]) == 4
# assert exercise_longest_ap_subarray([5]) == 1

### Gợi ý ba mức

**Bài 1** — M1: sort trước. M2: `[0, 5, 10, 15, 20]`. M3: mọi hiệu bằng 5.

**Bài 2** — M1: công sai lấy từ hai phần tử đầu. M2: `d = s[1] - s[0]`. M3: `for i in range(2, len(s)): if s[i] - s[i-1] != d: return False`, cuối cùng `return True`.

**Bài 3** — M1: không sort. M2: `d = arr[1] - arr[0]`. M3: `return all(b - a == d for a, b in zip(arr, arr[1:]))`.

**Bài 4** — M1: cấp số nhân dương là dãy đơn điệu → sort cũng đủ. M2: tỉ số bằng nhau `s[i]/s[i-1] == s[1]/s[0]` ⇔ `s[i] * s[0] == s[i-1] * s[1]`. M3: kiểm tra phép nhân chéo cho mọi `i >= 2`.

**Bài 5** — M1: xử lý `lo == hi` đầu tiên. M2: `(hi - lo) % (n - 1)` phải 0. M3: với mỗi `x`, `(x - lo) % d == 0` và `x` chưa trong `seen`.

**Bài 6** — M1: một đoạn AP bị "đứt" khi hiệu mới khác hiệu trước. M2: giữ `cur` = độ dài đoạn AP kết thúc tại `i`; nếu `arr[i]-arr[i-1] == arr[i-1]-arr[i-2]` thì `cur += 1` ngược lại `cur = 2`. M3: khởi tạo `best = cur = min(len(arr), 2)` và duyệt `i` từ 2.

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [24]:
def answer_by_hand():
    return (True, 5)

def answer_can_make_ap(arr):
    s = sorted(arr)
    d = s[1] - s[0]
    for i in range(2, len(s)):
        if s[i] - s[i - 1] != d:
            return False
    return True

def answer_is_ap_as_is(arr):
    d = arr[1] - arr[0]
    return all(b2 - a2 == d for a2, b2 in zip(arr, arr[1:]))

def answer_can_make_gp(arr):
    s = sorted(arr)
    return all(s[i] * s[0] == s[i - 1] * s[1] for i in range(2, len(s)))

def answer_can_make_ap_linear(arr):
    n = len(arr)
    lo, hi = min(arr), max(arr)
    if lo == hi:
        return True
    if (hi - lo) % (n - 1):
        return False
    d = (hi - lo) // (n - 1)
    seen = set()
    for x in arr:
        if (x - lo) % d or x in seen:
            return False
        seen.add(x)
    return True

def answer_longest_ap_subarray(arr):
    n = len(arr)
    if n <= 2:
        return n
    best = cur = 2
    for i in range(2, n):
        if arr[i] - arr[i - 1] == arr[i - 1] - arr[i - 2]:
            cur += 1
        else:
            cur = 2
        best = max(best, cur)
    return best

assert answer_by_hand() == (True, 5) and can_make_ap_sort([20, 5, 15, 10, 0])
assert answer_can_make_ap([3, 5, 1]) is True and answer_can_make_ap([1, 2, 4]) is False
assert answer_is_ap_as_is([5, 3, 1]) is True and answer_is_ap_as_is([3, 5, 1]) is False
assert answer_can_make_gp([4, 1, 16, 2, 8]) is True
assert answer_can_make_gp([1, 2, 3]) is False
assert answer_can_make_gp([3, 3, 3]) is True
assert answer_can_make_ap_linear([1, 7, 4, 10]) is True
assert answer_can_make_ap_linear([1, 3, 3, 7]) is False
assert answer_can_make_ap_linear([2, 2]) is True
assert answer_longest_ap_subarray([1, 3, 5, 7, 2, 4, 6]) == 4
assert answer_longest_ap_subarray([10, 7, 4, 6, 8, 10, 11]) == 4
assert answer_longest_ap_subarray([5]) == 1
def brute_longest(arr):
    n = len(arr); best = min(n, 2)
    for i in range(n):
        for j in range(i + 1, n):
            if answer_is_ap_as_is(arr[i:j + 1]):
                best = max(best, j - i + 1)
    return best
for _ in range(500):
    arr = [random.randint(-3, 3) for _ in range(random.randint(1, 10))]
    assert answer_longest_ap_subarray(arr) == brute_longest(arr), arr
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def canMakeArithmeticProgression(self, arr: list[int]) -> bool:
        s = sorted(arr)                    # AP luôn đơn điệu -> thử thứ tự đã sort
        d = s[1] - s[0]
        return all(s[i] - s[i - 1] == d for i in range(2, len(s)))
```

**Checklist trước khi submit:**
- [ ] Đã **sort** (hoặc dùng cách min/max) trước khi so hiệu?
- [ ] Vòng lặp không đọc vượt biên?
- [ ] Đã xét trường hợp mọi số bằng nhau (`d = 0`)?
- [ ] Nếu dùng cách `O(n)`: kiểm tra chia hết, nằm trên lưới, **không trùng**?
- [ ] Không chia cho 0, không so sánh float?